# Loading the Dataset
In this section, we will load the dataset and ensure there are no duplicate entries.

In [4]:
import pandas as pd

# Load the dataset
dataset_path = "WELFake_Dataset.csv"
data = pd.read_csv(dataset_path)

# Remove duplicates
data = data.drop_duplicates()

# Display the number of rows and the first few rows of the dataset
print(f"Number of rows after removing duplicates: {len(data)}")
data.head()

Number of rows after removing duplicates: 72134


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


# Fine-tuning BERT and RoBERTa
In this section, we will fine-tune both BERT and RoBERTa models. The goal is to put the two models side by side to see which shows better performance on the dataset and continue to the explanations using the better, or both, models.

This is how we will be proceding with this part of the pipelins:

* Tokenization -> We use both BERT and RoBERTa tokenizers to convert raw text into meaningful vectors using attention mechanism.
* Pass tokens through BERT and RoBERTa -> Produces these vectors which we will be using for classification.
* Add a classifier head -> Add a neural layer on top of BERT and RoBERTa. 
* Fine-tuning -> Both BERT and RoBERTa are pretrained on a massive corpora to understand language. Our Classifier head is randomly initialized. In this step we will be training the classifier and also update the weight of BERT and RoBERTa to fit our downstram task.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import torch

# Reproducibility and device setup
torch.manual_seed(42)
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Split the dataset into train and test (80/20)
data['label'] = data['label'].astype(int)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['text'].astype(str),
    data['label'],
    test_size=0.2,
    random_state=42,
    stratify=data['label']
 )

def tokenize_data(tokenizer, texts, max_length=256):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

# Tokenizers
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Tokenize datasets
train_encodings_bert = tokenize_data(bert_tokenizer, train_texts)
test_encodings_bert = tokenize_data(bert_tokenizer, test_texts)
train_encodings_roberta = tokenize_data(roberta_tokenizer, train_texts)
test_encodings_roberta = tokenize_data(roberta_tokenizer, test_texts)

train_labels_tensor = torch.tensor(train_labels.values, dtype=torch.long)
test_labels_tensor = torch.tensor(test_labels.values, dtype=torch.long)

class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: tensor[idx] for key, tensor in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset_bert = FakeNewsDataset(train_encodings_bert, train_labels_tensor)
test_dataset_bert = FakeNewsDataset(test_encodings_bert, test_labels_tensor)
train_dataset_roberta = FakeNewsDataset(train_encodings_roberta, train_labels_tensor)
test_dataset_roberta = FakeNewsDataset(test_encodings_roberta, test_labels_tensor)

batch_size = 8
train_loader_bert = DataLoader(train_dataset_bert, batch_size=batch_size, shuffle=True)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=batch_size)
train_loader_roberta = DataLoader(train_dataset_roberta, batch_size=batch_size, shuffle=True)
test_loader_roberta = DataLoader(test_dataset_roberta, batch_size=batch_size)

Using device: mps


In [9]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    preds = []
    labels = []

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

        batch_preds = torch.argmax(outputs.logits, dim=1)
        preds.extend(batch_preds.cpu().numpy())
        labels.extend(batch['labels'].cpu().numpy())

    return preds, labels, total_loss / len(dataloader)

In [10]:
import time

# Fine-tune BERT with pure PyTorch loop
bert_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
optimizer_bert = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 3

print("BERT Classifier Head Details:")
print(bert_model.classifier)
print("\nStarting BERT fine-tuning...")

start_time = time.time()
for epoch in range(num_epochs):
    train_loss = train_epoch(bert_model, train_loader_bert, optimizer_bert, device)
    _, _, val_loss = evaluate_model(bert_model, test_loader_bert, device)
    print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

end_time = time.time()
print(f"BERT fine-tuning completed in {end_time - start_time:.2f} seconds.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT Classifier Head Details:
Linear(in_features=768, out_features=2, bias=True)

Starting BERT fine-tuning...


  0%|          | 0/7214 [00:00<?, ?it/s]

Epoch 1/3 | Train Loss: 0.0531 | Val Loss: 0.0486


  0%|          | 0/7214 [00:00<?, ?it/s]

Epoch 2/3 | Train Loss: 0.0191 | Val Loss: 0.0221


  0%|          | 0/7214 [00:00<?, ?it/s]

Epoch 3/3 | Train Loss: 0.0112 | Val Loss: 0.0352
BERT fine-tuning completed in 11544.06 seconds.


In [11]:
bert_preds, bert_true, bert_eval_loss = evaluate_model(bert_model, test_loader_bert, device)
print(f"BERT Test Loss: {bert_eval_loss:.4f}")
bert_report = classification_report(bert_true, bert_preds)
print("\nBERT Classification Report:\n", bert_report)


BERT Test Loss: 0.0352

BERT Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99      7006
           1       1.00      0.98      0.99      7421

    accuracy                           0.99     14427
   macro avg       0.99      0.99      0.99     14427
weighted avg       0.99      0.99      0.99     14427



In [15]:
import os
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset

# Auto-setup device if missing
if "device" not in globals():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
print(f"Using device: {device}")

# Load model/tokenizer if not already in memory
if "bert_tokenizer" not in globals() or "bert_model" not in globals():
    candidate_model_dirs = [
        "/Users/bahaar/Desktop/NLP/bert_welfake_model",
        "bert_welfake_model",
    ]
    model_dir = next((p for p in candidate_model_dirs if os.path.isdir(p)), None)
    if model_dir is None:
        raise FileNotFoundError(
            "Could not find saved model directory 'bert_welfake_model'. "
            "Run the training/saving cell first or place the folder at /Users/bahaar/Desktop/NLP/bert_welfake_model"
        )

    bert_tokenizer = BertTokenizer.from_pretrained(model_dir)
    bert_model = BertForSequenceClassification.from_pretrained(model_dir).to(device)
    print(f"Loaded BERT model from: {model_dir}")

# Paths provided by user
true_path = "/Users/bahaar/Desktop/NLP/True.csv"
fake_path = "/Users/bahaar/Desktop/NLP/Fake.csv"

# Load data
true_df = pd.read_csv(true_path)
fake_df = pd.read_csv(fake_path)

# Robust text extraction (prefers `text`, otherwise combines available text-like columns)
def build_text_series(df):
    if "text" in df.columns:
        return df["text"].astype(str)

    text_like_cols = [c for c in ["title", "text", "content", "article"] if c in df.columns]
    if text_like_cols:
        return df[text_like_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip()

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if not object_cols:
        raise ValueError("No usable text columns found in dataframe.")
    return df[object_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip()

true_texts = build_text_series(true_df)
fake_texts = build_text_series(fake_df)

# Balanced random sampling: 5000 from each class
n_per_class = 5000
if len(true_texts) < n_per_class or len(fake_texts) < n_per_class:
    raise ValueError(
        f"Not enough rows for balanced sampling. True={len(true_texts)}, Fake={len(fake_texts)}, required={n_per_class} each."
    )

true_sample = pd.DataFrame({"text": true_texts}).sample(n=n_per_class, random_state=42).copy()
true_sample["label"] = 1  # True news

fake_sample = pd.DataFrame({"text": fake_texts}).sample(n=n_per_class, random_state=42).copy()
fake_sample["label"] = 0  # Fake news

balanced_test_df = pd.concat([true_sample, fake_sample], ignore_index=True)
balanced_test_df = balanced_test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced external test set created.")
print("Class counts:")
print(balanced_test_df["label"].value_counts().sort_index())
print(f"Total rows: {len(balanced_test_df)}")

# Tokenize and build dataloader for BERT
ext_texts = balanced_test_df["text"].astype(str)
ext_labels = torch.tensor(balanced_test_df["label"].values, dtype=torch.long)
ext_encodings = bert_tokenizer(
    ext_texts.tolist(),
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt",
)

class ExternalEvalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

external_dataset = ExternalEvalDataset(ext_encodings, ext_labels)
external_loader = DataLoader(external_dataset, batch_size=16)

# Evaluate BERT on external balanced set
bert_model.eval()
all_preds, all_true, all_probs = [], [], []
total_loss = 0.0

with torch.no_grad():
    for batch in external_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = bert_model(**batch)

        total_loss += outputs.loss.item()

        probs = torch.softmax(outputs.logits, dim=1)[:, 1]
        preds = torch.argmax(outputs.logits, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(batch["labels"].cpu().numpy())

avg_loss = total_loss / len(external_loader)
accuracy = accuracy_score(all_true, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_true, all_preds, average="binary", zero_division=0
)
auc = roc_auc_score(all_true, all_probs)
cm = confusion_matrix(all_true, all_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("\n===== BERT Evaluation on External Balanced Test Set =====")
print(f"Loss: {avg_loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (True class): {precision:.4f}")
print(f"Recall (True class): {recall:.4f}")
print(f"F1-score (True class): {f1:.4f}")
print(f"Specificity (Fake class recall): {specificity:.4f}")
print(f"ROC-AUC: {auc:.4f}")

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(cm)

print("\nDetailed Classification Report:")
print(
    classification_report(
        all_true,
        all_preds,
        target_names=["Fake (0)", "True (1)"],
        digits=4,
        zero_division=0,
    )
)

# Save sampled test set for reproducibility
balanced_test_df.to_csv("/Users/bahaar/Desktop/NLP/balanced_external_test_10k.csv", index=False)
print("Saved sampled test set to /Users/bahaar/Desktop/NLP/balanced_external_test_10k.csv")

Using device: mps
Balanced external test set created.
Class counts:
label
0    5000
1    5000
Name: count, dtype: int64
Total rows: 10000

===== BERT Evaluation on External Balanced Test Set =====
Loss: 9.4956
Accuracy: 0.0005
Precision (True class): 0.0000
Recall (True class): 0.0000
F1-score (True class): 0.0000
Specificity (Fake class recall): 0.0010
ROC-AUC: 0.0000

Confusion Matrix [[TN, FP], [FN, TP]]:
[[   5 4995]
 [5000    0]]

Detailed Classification Report:
              precision    recall  f1-score   support

    Fake (0)     0.0010    0.0010    0.0010      5000
    True (1)     0.0000    0.0000    0.0000      5000

    accuracy                         0.0005     10000
   macro avg     0.0005    0.0005    0.0005     10000
weighted avg     0.0005    0.0005    0.0005     10000

Saved sampled test set to /Users/bahaar/Desktop/NLP/balanced_external_test_10k.csv


In [16]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_auc_score

# Diagnostic: check if model's label mapping is opposite of (Fake=0, True=1)
flipped_preds = [1 - p for p in all_preds]
flipped_probs = [1.0 - p for p in all_probs]

acc_flip = accuracy_score(all_true, flipped_preds)
prec_flip, rec_flip, f1_flip, _ = precision_recall_fscore_support(
    all_true, flipped_preds, average="binary", zero_division=0
)
auc_flip = roc_auc_score(all_true, flipped_probs)
cm_flip = confusion_matrix(all_true, flipped_preds)

print("===== Diagnostic Report with Flipped Prediction Labels =====")
print(f"Accuracy: {acc_flip:.4f}")
print(f"Precision (True class): {prec_flip:.4f}")
print(f"Recall (True class): {rec_flip:.4f}")
print(f"F1-score (True class): {f1_flip:.4f}")
print(f"ROC-AUC: {auc_flip:.4f}")
print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(cm_flip)
print("\nDetailed Classification Report:")
print(
    classification_report(
        all_true,
        flipped_preds,
        target_names=["Fake (0)", "True (1)"],
        digits=4,
        zero_division=0,
    )
)

===== Diagnostic Report with Flipped Prediction Labels =====
Accuracy: 0.9995
Precision (True class): 0.9990
Recall (True class): 1.0000
F1-score (True class): 0.9995
ROC-AUC: 1.0000

Confusion Matrix [[TN, FP], [FN, TP]]:
[[4995    5]
 [   0 5000]]

Detailed Classification Report:
              precision    recall  f1-score   support

    Fake (0)     1.0000    0.9990    0.9995      5000
    True (1)     0.9990    1.0000    0.9995      5000

    accuracy                         0.9995     10000
   macro avg     0.9995    0.9995    0.9995     10000
weighted avg     0.9995    0.9995    0.9995     10000



In [20]:
import os
import torch

save_dir = "/Users/bahaar/Desktop/NLP"
os.makedirs(save_dir, exist_ok=True)

# Move tensors to CPU for portable checkpoints
state_dict_cpu = {k: v.detach().cpu() for k, v in bert_model.state_dict().items()}

pt_path = os.path.join(save_dir, "bert_welfake.pt")
pth_path = os.path.join(save_dir, "bert_welfake.pth")

# Use legacy (non-zip) serialization to avoid the torch zip-writer issue in this environment
torch.save(state_dict_cpu, pt_path, _use_new_zipfile_serialization=False)
torch.save(state_dict_cpu, pth_path, _use_new_zipfile_serialization=False)

print(f"Saved .pt checkpoint to: {pt_path}")
print(f"Saved .pth checkpoint to: {pth_path}")

Saved .pt checkpoint to: /Users/bahaar/Desktop/NLP/bert_welfake.pt
Saved .pth checkpoint to: /Users/bahaar/Desktop/NLP/bert_welfake.pth


In [3]:
import os
import pandas as pd
import torch
import torch.utils._pytree as _torch_pytree

# Compatibility shim for older torch builds used by transformers imports
def _register_pytree_node_compat(typ, flatten_fn, unflatten_fn, *args, **kwargs):
    return _torch_pytree._register_pytree_node(typ, flatten_fn, unflatten_fn)

_torch_pytree.register_pytree_node = _register_pytree_node_compat

from transformers import BertTokenizer

if "data" not in globals():
    dataset_path = "WELFake_Dataset.csv"
    data = pd.read_csv(dataset_path)

if "bert_tokenizer" not in globals():
    bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Add BERT tokens to the dataset
data['bert_tokens'] = data['text'].astype(str).apply(
    lambda x: bert_tokenizer(
        x,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )['input_ids'].tolist()
)

# Save the updated dataset
data.to_csv("WELFake_Dataset_with_bert_tokens.csv", index=False)